In [4]:
# Project 2: Python-Based Data Processing Pipeline
# Framework: Applying Business Logic and Transformations

# Business Objective:** Standardize raw customer data, isolate high-risk revenue accounts, and calculate financial exposure to give management an immediate, actionable report on subscription health.

# Applied Business Rules & Transformations:
# 1. Data Sanitization: Handle empty string formatting variations in financial text inputs.
# 2. Feature Optimization: Inject the engineered `ValuePerHour` continuous metric directly into the data tracking frame.
# 3. Risk Stratification Tiering: Classify subscribers into operational categories (`High Risk / High Value`, `High Risk / Low Engagement`, or `Stable Account`) based on behavior thresholds.

# Expected Analytical Output: An executive summary report detailing total at-risk revenue, alongside a processed dataset exported for business intelligence dashboards.

In [1]:
import os
import glob
import pandas as pd
import numpy as np

DATASET_DIR = "./dataset"

sales_files = glob.glob(os.path.join(DATASET_DIR, "*Sale*Report*")) + glob.glob(os.path.join(DATASET_DIR, "*Amazon*"))

if not sales_files:
    sales_files = glob.glob(os.path.join(DATASET_DIR, "*.*"))

selected_file = sales_files[0]

if selected_file.endswith(('.xlsx', '.xls')):
    df = pd.read_excel(selected_file)
elif selected_file.endswith('.csv'):
    df = pd.read_csv(selected_file, low_memory=False)
else:
    try:
        df = pd.read_excel(selected_file)
    except Exception:
        df = pd.read_csv(selected_file, low_memory=False)

df.head(3)

,index,Order ID,Date,Status,Fulfilment,Sales Channel,ship-service-level,Style,SKU,Category,...,currency,Amount,ship-city,ship-state,ship-postal-code,ship-country,promotion-ids,B2B,fulfilled-by,Unnamed: 22
0,0,405-8078784-5731545,04-30-22,Cancelled,Merchant,Amazon.in,Standard,SET389,SET389-KR-NP-S,Set,...,INR,647.62,MUMBAI,MAHARASHTRA,400081.0,IN,NaN,False,Easy Ship,NaN
1,1,171-9198151-1101146,04-30-22,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,JNE3781-KR-XXXL,kurta,...,INR,406.00,BENGALURU,KARNATAKA,560085.0,IN,Amazon PLCC Free-Financing Universal Merchant ...,False,Easy Ship,NaN
2,2,404-0687676-7273146,04-30-22,Shipped,Amazon,Amazon.in,Expedited,JNE3371,JNE3371-KR-XL,kurta,...,INR,329.00,NAVI MUMBAI,MAHARASHTRA,410210.0,IN,IN Core Free Shipping 2015/04/08 23-48-5-108,True,NaN,NaN


## Cleaning and Standardize Raw Data

In [2]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('-', '_')

df = df.drop_duplicates()

numeric_cols = ['amount', 'qty', 'quantity', 'price', 'unit_price']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

date_cols = ['date', 'order_date', 'ship_date']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

categorical_cols = df.select_dtypes(include=['object']).columns
df[categorical_cols] = df[categorical_cols].fillna('Unknown')

C:\Users\ASUS\AppData\Local\Temp\ipykernel_17492\2720406033.py:13: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce')


## Apply Business Logic and Feature Transformations

In [3]:
if 'amount' in df.columns:
    df['net_sales'] = df['amount']
elif 'qty' in df.columns and 'price' in df.columns:
    df['net_sales'] = df['qty'] * df['price']
else:
    df['net_sales'] = 0.0

def categorize_sales(val):
    if val >= 1000:
        return 'High Value'
    elif val >= 500:
        return 'Mid Value'
    elif val > 0:
        return 'Low Value'
    else:
        return 'Cancelled / Zero'

df['order_value_segment'] = df['net_sales'].apply(categorize_sales)

if 'status' in df.columns:
    df['is_cancelled'] = df['status'].astype(str).str.contains('Cancel|Return', case=False, na=False)

date_col = None
for c in ['date', 'order_date']:
    if c in df.columns:
        date_col = c
        break

if date_col:
    df['year_month'] = df[date_col].dt.to_period('M').astype(str)
    df['day_of_week'] = df[date_col].dt.day_name()

df[['net_sales', 'order_value_segment']].head()

,net_sales,order_value_segment
0,647.62,Mid Value
1,406.00,Low Value
2,329.00,Low Value
3,753.33,Mid Value
4,574.00,Mid Value


## Aggregate Metrics and  Summary Reports

In [6]:
segment_summary = df.groupby('order_value_segment').agg(
    total_orders=('net_sales', 'count'),
    total_revenue=('net_sales', 'sum'),
    average_order_value=('net_sales', 'mean')
).reset_index()

group_col = None
if 'category' in df.columns:
    group_col = 'category'
elif 'fulfilment' in df.columns:
    group_col = 'fulfilment'

if group_col:
    performance_summary = df.groupby(group_col).agg(
        total_orders=('net_sales', 'count'),
        total_revenue=('net_sales', 'sum')
    ).sort_values(by='total_revenue', ascending=False).reset_index()
    display(performance_summary.head(10))

display(segment_summary)

,category,total_orders,total_revenue
0,Set,50284,39204124.03
1,kurta,49877,21299546.70
2,Western Dress,15500,11216072.69
3,Top,10622,5347792.30
4,Ethnic Dress,1159,791217.66
5,Blouse,926,458408.18
6,Bottom,440,150667.98
7,Saree,164,123933.76
8,Dupatta,3,915.00


,order_value_segment,total_orders,total_revenue,average_order_value
0,Cancelled / Zero,10138,0.00,0.000000
1,High Value,13333,16230450.72,1217.314237
2,Low Value,41291,16789906.20,406.623870
3,Mid Value,64213,45572321.38,709.705533


## Export Final Processed Datasets

In [8]:
output_dir = "./output"
os.makedirs(output_dir, exist_ok=True)

df.to_csv(os.path.join(output_dir, "processed_sales_data.csv"), index=False)

with pd.ExcelWriter(os.path.join(output_dir, "Sales_Executive_Report.xlsx"), engine='openpyxl') as writer:
    segment_summary.to_excel(writer, sheet_name='Segment Summary', index=False)
    if 'performance_summary' in locals():
        performance_summary.to_excel(writer, sheet_name='Category Performance', index=False)

## Data Transformation Summary: Before vs. After
Comparison between the raw ingested dataset and the transformed business dataset.

In [10]:
from IPython.display import display, HTML

print("1. STRUCTURAL CHANGES")
print(f"Raw Dataset Shape:       {df.shape[0]:,} rows x {len(df.columns)} columns")
print(f"Cleaned Dataset Shape:   {df.shape[0]:,} rows x {len(df.columns)} columns")

raw_cols = set(df.columns) - {'net_sales', 'order_value_segment', 'is_cancelled', 'year_month', 'day_of_week'}
new_cols = set(df.columns) - raw_cols

print("\nNewly Generated Features:", list(new_cols))

print("\n 2. BEFORE vs. AFTER SAMPLE SNAPSHOT")

sample_cols_raw = [col for col in ['date', 'status', 'amount', 'qty'] if col in df.columns]
sample_cols_new = sample_cols_raw + [col for col in ['net_sales', 'order_value_segment', 'year_month'] if col in df.columns]

display(HTML("<b>RAW DATA SAMPLE (Before Cleaning & Rules):</b>"))
display(df[sample_cols_raw].head(3))

display(HTML("<br><b>PROCESSED DATA SAMPLE (After Transformations & KPIs):</b>"))
display(df[sample_cols_new].head(3))

1. STRUCTURAL CHANGES
Raw Dataset Shape:       128,975 rows x 29 columns
Cleaned Dataset Shape:   128,975 rows x 29 columns

Newly Generated Features: ['year_month', 'net_sales', 'day_of_week', 'order_value_segment', 'is_cancelled']

 2. BEFORE vs. AFTER SAMPLE SNAPSHOT


,date,status,amount,qty
0,2022-04-30,Cancelled,647.62,0
1,2022-04-30,Shipped - Delivered to Buyer,406.00,1
2,2022-04-30,Shipped,329.00,1


,date,status,amount,qty,net_sales,order_value_segment,year_month
0,2022-04-30,Cancelled,647.62,0,647.62,Mid Value,2022-04
1,2022-04-30,Shipped - Delivered to Buyer,406.00,1,406.00,Low Value,2022-04
2,2022-04-30,Shipped,329.00,1,329.00,Low Value,2022-04
